In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Muat model sentimen Bahasa Indonesia pre-trained:

from transformers import pipeline

# Inisialisasi pipeline model sentimen Bahasa Indonesia
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    truncation=True,
    max_length=512
)

# Uji coba pada 1 kalimat
hasil_uji = sentiment_analyzer("Aplikasi Alfagift sangat membantu!")
print("Hasil Uji Coba Model:", hasil_uji)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Hasil Uji Coba Model: [{'label': 'positive', 'score': 0.997519314289093}]


In [ ]:
#Terapkan model pada 10 baris pertama dari dataset

import pandas as pd

# Membaca dataset dari Google Drive
file_path = '/content/drive/MyDrive/Data Analis Angkatan 2 Tahun 2026_Faneza Nur Mardania/21 Agustus 2026_Pertemuan 26/File Tugas/data_labeled.csv'

df = pd.read_csv(file_path)   # dari Tugas 2
sample = df.head(10).copy()

hasil = sentiment_analyzer(sample['teks'].astype(str).tolist())
sample['prediksi_bert'] = [h['label'] for h in hasil]
sample['confidence']    = [round(h['score'], 3) for h in hasil]

print(sample[['teks', 'sentimen', 'prediksi_bert', 'confidence']])

                                                teks sentimen prediksi_bert  \
0  woi cs bales komplain gw di alfagift, pesenan ...  negatif      negative   
1  bagus tapi sekarang poin kecil dan kupon ngaco...  negatif      negative   
2           harga jauh lebih mahal dari toko sebelah  negatif      negative   
3  lambat bagaikan siput pengantaran kurirnya, ra...  negatif      negative   
4                                             mantap  positif      positive   
5  pengembalian dana nya buruk, dana tidak pernah...  negatif      negative   
6  Apk Scan nih ( penipuan) masa katanya bisa bel...  negatif      negative   
7  maaf saya kasih 1 biar di notice. tolong buat ...  negatif      negative   
8  transaksi pick up pada 17 Agustus 2026, dan su...  negatif      negative   
9  jujur padahal alfanya deket, cmn aku tetep pak...  positif      positive   

   confidence  
0       0.905  
1       0.855  
2       0.986  
3       0.985  
4       0.976  
5       0.999  
6       0.998  
7 

In [ ]:
# Bandingkan akurasi DistilBERT dengan model dari Tugas 5

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Gunakan split yang sama seperti Tugas 3
X = df['teks']  # gunakan teks ASLI (DistilBERT punya tokenizer sendiri)
y = df['sentimen']
_, X_test_raw, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Prediksi pada test set
hasil_bert = sentiment_analyzer(X_test_raw.tolist())
pred_bert  = [h['label'] for h in hasil_bert]

# Sesuaikan label (model output mungkin 'positive'/'negative', dataset mungkin 'positif'/'negatif')
mapping = {'positive': 'positif', 'negative': 'negatif', 'neutral': 'netral'}
pred_bert_map = [mapping.get(p.lower(), p) for p in pred_bert]

print('=== DistilBERT/RoBERTa Indonesia ===')
print(f'Akurasi: {accuracy_score(y_test, pred_bert_map):.4f}')
print('\n', classification_report(y_test, pred_bert_map))


=== DistilBERT/RoBERTa Indonesia ===
Akurasi: 0.5420

               precision    recall  f1-score   support

     negatif       0.42      0.96      0.58       284
      netral       0.54      0.12      0.19       313
     positif       0.84      0.57      0.68       403

    accuracy                           0.54      1000
   macro avg       0.60      0.55      0.49      1000
weighted avg       0.63      0.54      0.50      1000

